# DEEP-GUARD — Public Colab Training & Recovery

This notebook clones the public [DEEP-GUARD repository](https://github.com/ABHILATTHE6/DEEP-GUARD), reconstructs a reproducible Tiny-GenImage baseline, resumes training from a GitHub Release checkpoint when available, publishes checkpoint progress, evaluates the held-out test split, and commits the final report back to GitHub.

**Use a GPU runtime:** Runtime → Change runtime type → T4 GPU (or another available CUDA GPU).

**Data note:** The dataset comes from [TheKernel01/Tiny-GenImage](https://huggingface.co/datasets/TheKernel01/Tiny-GenImage), whose current dataset card lists CC-BY-NC-SA 4.0. Review and cite the source terms. This recovery baseline is *not* the earlier mixed Tiny-GenImage + COCO + Nature V3 split; label the final experiment accurately. The notebook does not upload raw images to GitHub.

**Persistence:** Colab's local disk can reset. Model checkpoints are synced to public GitHub Release assets after every epoch; a final evaluation report and dataset manifest are committed to the repo. GitHub Release model files are public downloads, so do not put any private data into them.

## 1. Clone and install

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

REPO_DIR = Path("/content/DEEP-GUARD")
if not (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", "https://github.com/ABHILATTHE6/DEEP-GUARD.git", str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)
print("Repository:", REPO_DIR)
print("Python:", sys.version)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".", "datasets>=3,<5", "pillow>=10,<12", "scikit-learn>=1.5,<2", "requests>=2.31,<3"],
    check=True,
)
subprocess.run([sys.executable, "-m", "compileall", "-q", "backend", "ml", "scripts"], check=True)
import torch
print("Python syntax check: passed")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected. Switch the Colab runtime to GPU before full training.")

## 2. Enter a GitHub token so training can save to the public repository

Checkpoint upload requires a fine-grained GitHub token limited to **ABHILATTHE6/DEEP-GUARD** with **Contents: Read and write** permission. Create it from [GitHub token settings](https://github.com/settings/personal-access-tokens/new). Paste it into the hidden prompt below; the value is not written into the notebook or printed. Do not share or commit the token.

You may skip the token prompt, but checkpoints then remain only on Colab's temporary disk and will not survive a reset. For the final deadline run, enter the token.

In [ ]:
from getpass import getpass

GH_TOKEN_VALUE = getpass("Fine-grained GitHub token (input is hidden; required for persistent checkpoint uploads): ").strip()
if GH_TOKEN_VALUE:
    os.environ["GH_TOKEN"] = GH_TOKEN_VALUE
    print("Token loaded into this runtime (value not displayed).")
else:
    os.environ.pop("GH_TOKEN", None)
    print("WARNING: checkpoint uploads disabled; local training files will be lost if this runtime resets.")

## 3. Rebuild the dataset

This cell uses a seeded stream and can continue a partial preparation under the same output directory. The quick-start preset uses 3,000 AI + 3,000 real train images, 400 per class for validation, and 400 per class for test. It trains for 5 epochs to prioritize finishing the deadline run; change the notebook's --epochs value to 10 if time permits. Downloading and preparing the source may take several minutes or longer; let it finish.

In [ ]:
DATA_ROOT = Path("/content/deepguard-tiny")
prepare_cmd = [
    sys.executable, "scripts/prepare_tiny_genimage_hf.py",
    "--output", str(DATA_ROOT),
    "--train-per-class", "3000",
    "--eval-per-class", "400",
    "--seed", "42",
    "--buffer-size", "4000",
]
subprocess.run(prepare_cmd, check=True)

for split in ("train", "val", "test"):
    print(f"\n{split.upper()}")
    for label in ("ai_generated", "real"):
        folder = DATA_ROOT / split / label
        print(label, "images:", sum(1 for p in folder.glob("*") if p.is_file()))

## 4. Restore the latest and best checkpoints from GitHub

On a new run, the release may not exist yet; the notebook then starts training from pretrained EfficientNet-B0. If a previous run published checkpoint assets, download them and resume from the latest saved epoch. The best checkpoint is downloaded separately for final evaluation.

In [ ]:
ARTIFACTS = REPO_DIR / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
LAST_CKPT = ARTIFACTS / "image_training_last.pt"
BEST_CKPT = ARTIFACTS / "image_training_best.pt"
SYNC_SCRIPT = REPO_DIR / "scripts" / "github_release_checkpoint.py"

def try_download(asset_name, destination):
    cmd = [
        sys.executable, str(SYNC_SCRIPT), "download",
        "--repo", "ABHILATTHE6/DEEP-GUARD",
        "--tag", "training-checkpoints",
        "--asset-name", asset_name,
        "--output", str(destination),
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode == 0:
        print(proc.stdout)
        return True
    print(f"No usable asset {asset_name} found yet. It is okay for a first run.")
    if proc.stderr:
        print(proc.stderr[-500:])
    return False

has_last = try_download("image-training-latest.pt", LAST_CKPT)
has_best = try_download("image-training-best.pt", BEST_CKPT)
RESUME_CKPT = LAST_CKPT if has_last else (BEST_CKPT if has_best else None)
print("Resume source:", RESUME_CKPT or "fresh pretrained start")

## 5. Train and continuously publish checkpoints

Training saves a local checkpoint after each completed epoch, then uploads it to the GitHub Release if a token was entered. The best checkpoint is selected by AI-positive validation ROC-AUC; the decision threshold is calibrated on validation data only to target the configured real-image false-positive rate. Do not tune using the test split.

For a fast first deadline run, keep 5–10 epochs. If a checkpoint from a previous run says epoch 10, the script will recognize that the total target is already reached and skip extra training.

In [ ]:
train_cmd = [
    sys.executable, "scripts/train_image.py",
    "--train", str(DATA_ROOT / "train"),
    "--val", str(DATA_ROOT / "val"),
    "--epochs", "5",
    "--batch-size", "64",
    "--num-workers", "2",
    "--lr", "0.0001",
    "--target-real-fpr", "0.03",
    "--output", str(LAST_CKPT),
    "--best-output", str(BEST_CKPT),
]
if RESUME_CKPT is not None:
    train_cmd += ["--resume-from", str(RESUME_CKPT)]
if os.environ.get("GH_TOKEN"):
    train_cmd += ["--publish-to-github"]
else:
    print("WARNING: GH_TOKEN absent; checkpoints will not be persisted to GitHub.")
subprocess.run(train_cmd, check=True)

# If a Release had a latest checkpoint but no best asset, use the latest as a fallback.
# Normally the best asset exists because the trainer publishes it whenever validation AUC improves.
if not BEST_CKPT.is_file() and LAST_CKPT.is_file():
    import shutil
    shutil.copy2(LAST_CKPT, BEST_CKPT)
    print("No best checkpoint asset found; copied the latest checkpoint for evaluation.")

## 6. Evaluate the held-out test set

The evaluation script selects/calibrates its real-probability threshold on the validation split, then evaluates once on the test split. The JSON report includes class mapping, AI-positive ROC-AUC, thresholded metrics, and the confusion matrix.

In [ ]:
EVAL_REPORT = ARTIFACTS / "image_evaluation.json"
eval_cmd = [
    sys.executable, "scripts/evaluate_image.py",
    "--checkpoint", str(BEST_CKPT),
    "--val", str(DATA_ROOT / "val"),
    "--test", str(DATA_ROOT / "test"),
    "--output", str(EVAL_REPORT),
    "--batch-size", "64",
    "--target-real-fpr", "0.03",
]
subprocess.run(eval_cmd, check=True)

## 7. Publish the evaluation report and dataset manifest to GitHub

The JSON report and small dataset manifest are committed under `reports/`. No images or secret tokens are committed. The model checkpoints remain downloadable from the public GitHub Release. The GitHub token must still be loaded for this step.

In [ ]:
import base64
import hashlib
import json
import requests
from datetime import datetime, timezone

if not os.environ.get("GH_TOKEN"):
    raise RuntimeError("GH_TOKEN is required to publish the final report to GitHub.")

report = json.loads(EVAL_REPORT.read_text(encoding="utf-8"))
report["dataset"] = "TheKernel01/Tiny-GenImage"
report["dataset_protocol"] = "train source split; source validation divided into disjoint val/test samples"
report["dataset_manifest_sha256"] = hashlib.sha256(
    (DATA_ROOT / "dataset_manifest.json").read_bytes()
).hexdigest()
report["checkpoint_sha256"] = hashlib.sha256(BEST_CKPT.read_bytes()).hexdigest()
report["checkpoint_asset"] = "image-training-best.pt"
report["repository"] = "https://github.com/ABHILATTHE6/DEEP-GUARD"
report["reproducibility_note"] = "Scores apply to this dataset protocol and should not be presented as the previous mixed V3 experiment."

manifest = json.loads((DATA_ROOT / "dataset_manifest.json").read_text(encoding="utf-8"))
repo_slug = "ABHILATTHE6/DEEP-GUARD"
headers = {
    "Accept": "application/vnd.github+json",
    "Authorization": f"Bearer {os.environ['GH_TOKEN']}",
    "X-GitHub-Api-Version": "2022-11-28",
}

def commit_json(repo_path, payload, message):
    api_url = f"https://api.github.com/repos/{repo_slug}/contents/{repo_path}"
    current = requests.get(api_url, headers=headers, timeout=30)
    body = {
        "message": message,
        "content": base64.b64encode(
            (json.dumps(payload, indent=2) + "\n").encode("utf-8")
        ).decode("ascii"),
        "branch": "main",
    }
    if current.status_code == 200:
        body["sha"] = current.json()["sha"]
    elif current.status_code != 404:
        current.raise_for_status()
    response = requests.put(api_url, headers=headers, json=body, timeout=60)
    response.raise_for_status()
    print("Committed", repo_path, "->", response.json().get("content", {}).get("html_url"))

commit_json(
    "reports/image_evaluation.json",
    report,
    "results: publish verified Tiny-GenImage baseline evaluation",
)
commit_json(
    "reports/dataset_manifest.json",
    manifest,
    "data: record reproducible Tiny-GenImage dataset manifest",
)
print("Report publication complete.")

## 8. Final checks

Confirm that the model checkpoint is present in the release and that the report is visible at [reports/](https://github.com/ABHILATTHE6/DEEP-GUARD/tree/main/reports). The browser/API service still needs a supported host for public interactive use; GitHub is the source repository and artifact home, not a permanent GPU runtime.

**For your report:** cite the dataset source, training configuration, split counts, calibrated threshold source, test metrics, and limitations. Do not claim that video or audio detection is trained unless those modalities have their own verified checkpoints and evaluation.